# Customer Segmentation Using RFM & K-Means

**Slab 1 – For Beginners**

This notebook is designed as a complete, reproducible submission. Run the cells from top to bottom.

## 1. Objective
Perform customer segmentation from transaction data using **RFM analysis**:

- **Recency:** how recently a customer purchased
- **Frequency:** how often the customer purchased
- **Monetary:** how much the customer spent

Then use K-Means clustering to identify actionable customer groups.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")

# UCI Online Retail dataset
DATA_URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

import io, zipfile, requests
content = requests.get(DATA_URL, timeout=120).content
with zipfile.ZipFile(io.BytesIO(content)) as z:
    excel_name = [n for n in z.namelist() if n.lower().endswith(".xlsx")][0]
    df = pd.read_excel(z.open(excel_name))

print("Original shape:", df.shape)
display(df.head())

## 2. Cleaning
The UCI dataset includes cancellations, missing Customer IDs and transactions that may not represent genuine positive purchases. For RFM segmentation we focus on identifiable customers and valid positive purchases.

In [ ]:
df.columns = df.columns.str.strip()

print("Missing values:")
display(df.isna().sum().sort_values(ascending=False))

print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().copy()
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

# Remove cancellations/returns and invalid transactions for customer-value segmentation.
df = df.dropna(subset=["CustomerID","InvoiceDate","Quantity","UnitPrice"]).copy()
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
df = df[~df["InvoiceNo"].astype(str).str.upper().str.startswith("C")].copy()

df["Revenue"] = df["Quantity"] * df["UnitPrice"]

print("Cleaned shape:", df.shape)
display(df[["Quantity","UnitPrice","Revenue"]].describe().round(2))

## 3. Exploratory transaction analysis

In [ ]:
monthly_sales = df.set_index("InvoiceDate").resample("M")["Revenue"].sum()
monthly_sales.plot(figsize=(12,5), marker="o", title="Monthly Revenue")
plt.ylabel("Revenue (£)")
plt.show()

country_sales = df.groupby("Country")["Revenue"].sum().sort_values(ascending=False).head(10)
display(country_sales.to_frame("Revenue"))
country_sales.sort_values().plot(kind="barh", figsize=(9,5), title="Top Countries by Revenue")
plt.xlabel("Revenue (£)")
plt.show()

top_products = df.groupby("Description")["Revenue"].sum().nlargest(10)
display(top_products.to_frame("Revenue"))

## 4. RFM calculation

In [ ]:
# Use one day after the latest transaction as the analysis reference date.
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (reference_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum")
)

display(rfm.describe().round(2))

# RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, col in zip(axes, ["Recency","Frequency","Monetary"]):
    sns.histplot(rfm[col], kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 5. Prepare data for K-Means
RFM variables are typically skewed, so we apply a log transformation before standardization. We select K using both the elbow/inertia curve and silhouette score.

In [ ]:
rfm_model = rfm.copy()

for col in ["Recency","Frequency","Monetary"]:
    rfm_model[col] = np.log1p(rfm_model[col])

scaler = StandardScaler()
X = scaler.fit_transform(rfm_model[["Recency","Frequency","Monetary"]])

inertias = []
silhouettes = []
ks = range(2, 9)

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, ax = plt.subplots()
ax.plot(list(ks), inertias, marker="o")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Inertia")
ax.set_title("Elbow Method")
plt.show()

fig, ax = plt.subplots()
ax.plot(list(ks), silhouettes, marker="o")
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Silhouette score")
ax.set_title("Silhouette Analysis")
plt.show()

best_k = list(ks)[int(np.argmax(silhouettes))]
print("Selected k based on highest silhouette score:", best_k)

## 6. Final K-Means segmentation

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
rfm["Cluster"] = kmeans.fit_predict(X)

segment_summary = rfm.groupby("Cluster").agg(
    Customers=("Recency","size"),
    Avg_Recency=("Recency","mean"),
    Avg_Frequency=("Frequency","mean"),
    Avg_Monetary=("Monetary","mean"),
    Total_Monetary=("Monetary","sum")
).round(2)

segment_summary["Customer_Pct"] = segment_summary["Customers"] / len(rfm) * 100
display(segment_summary.sort_values("Avg_Monetary", ascending=False))

# Visualize clusters in RFM space
plt.figure(figsize=(9,6))
sns.scatterplot(data=rfm, x="Recency", y="Monetary", hue="Cluster", palette="tab10", alpha=0.7)
plt.yscale("log")
plt.title("Customer Segments: Recency vs Monetary")
plt.show()

## 7. Segment interpretation and recommendations
Give each cluster a business-friendly name based on its measured averages.

Suggested interpretation logic:

- **Low recency + high frequency + high monetary:** Champions / VIP customers.
- **Low recency + moderate monetary:** Loyal or promising customers.
- **High recency + previously high monetary:** At-risk high-value customers.
- **High recency + low frequency + low monetary:** Hibernating / low-value customers.
- **Very recent but low frequency:** New customers.

Do not assign a label solely from the cluster number. Cluster numbers are arbitrary; use the RFM averages to determine the meaning.

Recommended actions:
- Champions: loyalty rewards, early access and premium service.
- Loyal customers: cross-sell and personalized recommendations.
- At-risk high-value customers: targeted win-back campaigns.
- New customers: onboarding and second-purchase incentives.
- Low-value/hibernating customers: low-cost automated campaigns rather than expensive incentives.